You previously answered how you would approach building a machine learning model to predict which NCAA players will become good NBA players. You are now given a dataset that contains publicly available data. The dataset `college_data.csv` contains 50 NBA players and their college stats. You'll notice that many of these players have multiple rows of data. This is because these players have multiple seasons of college data.

Your task is to create a dataframe using Python where each player only has one row of data that represents their entire college career.  Please put comments throughout your code to explain your thought process. Only use data from `college_data.csv` in your answer. 


**When you have finished your code, copy your code to the Python text box on your assessment.**


In [1]:
import pandas as pd

# Import data
df = pd.read_csv("college_data.csv")
display(df)


,Player_Name,Player_id,Age,Position,Height,Weight,Season_Number,Total_Minutes_Played,USG_Percentage,TOV_Percentage,OREB_Percentage,DREB_Percentage,AST_Percentage,STL_Percentage,BLK_Percentage
0,Alex Caruso,0,22.1,PG,77,188,4,1067,0.156478,0.258467,0.042344,0.099221,0.311484,0.042601,0.015509
1,Alex Caruso,0,21.1,PG,77,188,3,1038,0.190553,0.270651,0.028109,0.140754,0.348483,0.042056,0.002560
2,Alex Caruso,0,20.1,PG,77,188,2,1013,0.193751,0.235400,0.017365,0.121104,0.367084,0.043292,0.033405
3,Alex Caruso,0,19.1,PG,77,188,1,833,0.187925,0.269848,0.037924,0.129736,0.275589,0.050620,0.023838
4,Andre Roberson,1,21.3,PF,79,206,3,1036,0.198937,0.188718,0.105571,0.270531,0.094164,0.040123,0.043539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,Tyrese Haliburton,49,20.1,PG,77,175,2,806,0.201699,0.187808,0.045643,0.137495,0.353895,0.038798,0.019871
94,Tyrese Haliburton,49,19.1,PG,77,175,1,1163,0.091882,0.135633,0.027354,0.088743,0.173001,0.027593,0.028345
95,Victor Oladipo,50,20.9,SG,76,213,3,1021,0.221724,0.185018,0.117401,0.140149,0.151753,0.046426,0.028323
96,Victor Oladipo,50,19.9,SG,76,213,2,961,0.231846,0.177421,0.086926,0.153686,0.146785,0.031078,0.023138


In [2]:
print(df)

          Player_Name  Player_id   Age Position  Height  Weight  \
0         Alex Caruso          0  22.1       PG      77     188   
1         Alex Caruso          0  21.1       PG      77     188   
2         Alex Caruso          0  20.1       PG      77     188   
3         Alex Caruso          0  19.1       PG      77     188   
4      Andre Roberson          1  21.3       PF      79     206   
..                ...        ...   ...      ...     ...     ...   
93  Tyrese Haliburton         49  20.1       PG      77     175   
94  Tyrese Haliburton         49  19.1       PG      77     175   
95     Victor Oladipo         50  20.9       SG      76     213   
96     Victor Oladipo         50  19.9       SG      76     213   
97     Victor Oladipo         50  18.9       SG      76     213   

    Season_Number  Total_Minutes_Played  USG_Percentage  TOV_Percentage  \
0               4                  1067        0.156478        0.258467   
1               3                  1038      

In [4]:
#ONE ROW = ONE PLAYER
#Let's aggregate the data, but the aggregation will be different according to which variable we are taking into consideration
# For example,  'Total_Minutes_Played', 'USG_Percentage', 'TOV_Percentage', and other stats, we use the average respect the minutes played
# For phisical stats, we keep costant since it's the same (we keep the first or last according to the variable), same for individual information as name and position
#so what we need effectively is just one function to generate the KPI/minute player, the rest we take the 'first' value using dictionaire. 

def KPI_per_minutesplayed(player, minutes, kpi_variable):
    return (player[minutes] * player[kpi_variable]).sum() / player[minutes].sum()

#now the dictionary
aggregate_players_dict = {
    'Player_id': 'first',
    'Player_Name': 'first',        
    'Age': 'last',                 # the actual age
    'Position': 'last',           # the last position, if it's changed over time
    'Height': 'last',             #I use the last value, but we could also use the average height and weight over years
    'Weight': 'last',   
    'Season_Number' : 'last',      #how many season we have info 
    'Total_Minutes_Played': 'sum', # all minutes played over the season

    #NOW LET'S USE THE FUNCTION CREATED, player is the index we are using(we could also use player ID but we need another function), with the created variable of total minutes and 
    'USG_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'USG_Percentage'),
    'TOV_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'TOV_Percentage'),
    'OREB_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'OREB_Percentage'),
    'DREB_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'DREB_Percentage'),
    'AST_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'AST_Percentage'),
    'STL_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'STL_Percentage'),
    'BLK_Percentage': lambda x: KPI_per_minutesplayed(df.loc[x.index], 'Total_Minutes_Played', 'BLK_Percentage')
}

#create the aggregate DF
players_carreer_df = df.groupby('Player_id').agg(aggregate_players_dict).reset_index(drop = True)
#considerations: before the len of the df was 98, now it's 51 unique and distinct rows

print(f'college carreer stats! {players_carreer_df }')

college carreer stats!     Player_id              Player_Name   Age Position  Height  Weight  \
0           0              Alex Caruso  19.1       PG      77     188   
1           1           Andre Roberson  19.3       PF      79     206   
2           2            Anthony Davis  19.1       PF      82     222   
3           3              Bam Adebayo  19.7        C      82     243   
4           4              Ben Simmons  19.7       PF      82     239   
5           5             Bradley Beal  18.8       SG      77     202   
6           6            C.J. McCollum  19.5       PG      75     197   
7           7          Cameron Johnson  20.1        F      81     205   
8           8            Chris Boucher  23.2       PF      82     182   
9           9           Christian Wood  18.5       PF      83     216   
10         10           Damian Lillard  20.7       PG      75     189   
11         11        De'Anthony Melton  18.8       PG      75     193   
12         12          Dejou